# MiniMind Teaching Notebook

## From Deep Learning Basics to a Small Language Model System

This notebook is a formal, lecture-oriented introduction to the MiniMind project. It is written for students who already understand standard deep learning ideas such as neural networks, embeddings, hidden layers, and cross-entropy, but who are still new to large language models.

The goal is not to cover the entire repository. The goal is to make the central path of the project understandable, runnable, and teachable.

### Learning Objectives

By the end of this notebook, students should be able to:

- identify the main data formats used in MiniMind;
- explain how chat messages become token sequences through the tokenizer and chat template;
- understand why supervised fine-tuning masks non-assistant tokens;
- connect loss formulas to the actual source code implementation;
- describe the high-level structure of the MiniMind model and its training scripts.


## What Students Already Know, and What Is New Here

This notebook assumes students have just completed an introductory deep learning course. They should already be comfortable with:

- tensors, embeddings, hidden layers, and logits;
- cross-entropy loss and backpropagation;
- the idea that a neural network maps inputs to hidden representations and then to predictions.

What is new in a language model setting is mostly the **problem formulation**, not a completely different kind of learning:

- the input is a token sequence rather than an image or a tabular vector;
- the model predicts the **next token** at every position;
- the output space is the full vocabulary;
- structured dialogue data must first be serialized into one text prompt before tokenization.

A useful mental shift for beginners is:

> a causal language model is still a neural network trained with cross-entropy; the main differences are sequence structure, masking, and data formatting.


## Teaching Scope

This notebook focuses on the central path through the project:

1. repository map;
2. a minimal LLM mental model;
3. training data formats;
4. tokenizer behavior and special control tokens;
5. chat template construction;
6. dataset objects and what they return;
7. loss functions and their code implementation;
8. model skeleton;
9. training and inference entry points.

For students who are new to LLMs, Sections 1-8 form the **core path**. The MoE auxiliary loss and the DPO objective are included as **extension topics** rather than first-pass prerequisites.

A final real-inference cell is included but disabled by default so that the notebook remains lightweight during class.


In [9]:
from pathlib import Path
import sys
import json
import tempfile
import gc
import random

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

random.seed(7)
torch.manual_seed(7)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(7)

CANDIDATES = [Path.cwd(), Path('/home/student/work/LLM/minimind')]
REPO_ROOT = next(p for p in CANDIDATES if (p / 'model' / 'model_minimind.py').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dataset.lm_dataset import PretrainDataset, SFTDataset, DPODataset
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM


def print_json(obj):
    print(json.dumps(obj, ensure_ascii=False, indent=2))


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def show_snippet(path, start, end):
    path = Path(path)
    lines = path.read_text(encoding='utf-8').splitlines()
    for i in range(start - 1, min(end, len(lines))):
        print(f'{i + 1:>4}: {lines[i]}')


print('REPO_ROOT =', REPO_ROOT)
print('CUDA available =', torch.cuda.is_available())
print('Random seed = 7')


REPO_ROOT = /home/student/work/LLM/minimind
CUDA available = True
Random seed = 7


## 1. Repository Map

A helpful way to introduce MiniMind is to divide the repository into conceptual layers:

- **data layer**: how JSONL examples are stored and loaded;
- **tokenization/template layer**: how structured messages become a text sequence and then token IDs;
- **model layer**: the causal language model itself;
- **training layer**: scripts for pretraining and supervised fine-tuning;
- **application layer**: inference, API serving, and web demo.

The following cell lists the key files that define this instructional path.


In [10]:
key_files = {
    'dataset': REPO_ROOT / 'dataset' / 'lm_dataset.py',
    'model': REPO_ROOT / 'model' / 'model_minimind.py',
    'pretrain': REPO_ROOT / 'trainer' / 'train_pretrain.py',
    'sft': REPO_ROOT / 'trainer' / 'train_full_sft.py',
    'inference': REPO_ROOT / 'eval_llm.py',
}

for name, path in key_files.items():
    print(f'{name:>10} -> {path.relative_to(REPO_ROOT)}')


   dataset -> dataset/lm_dataset.py
     model -> model/model_minimind.py
  pretrain -> trainer/train_pretrain.py
       sft -> trainer/train_full_sft.py
 inference -> eval_llm.py


## 2. A Minimal LLM Mental Model

Before reading MiniMind-specific files, it helps to make the standard deep learning view explicit.

Given token IDs with shape $[B, T]$:

- the embedding layer maps them to $[B, T, H]$;
- the transformer stack keeps the same shape $[B, T, H]$;
- the language-model head maps hidden states to logits of shape $[B, T, V]$;
- cross-entropy compares these logits against shifted target token IDs.

This is the same general pattern students already know from other neural networks:

$$
\text{input} \rightarrow \text{hidden representation} \rightarrow \text{logits} \rightarrow \text{loss}
$$

The difference is that, for a causal LM, this happens **at every time step** in the sequence.


In [11]:
shape_cfg = MiniMindConfig(
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    vocab_size=6400,
    max_position_embeddings=128,
    use_moe=False,
)

shape_model = MiniMindForCausalLM(shape_cfg)
input_ids = torch.randint(0, shape_cfg.vocab_size, (2, 8))
embeddings = shape_model.model.embed_tokens(input_ids)
hidden_states, _, _ = shape_model.model(input_ids)
logits = shape_model.lm_head(hidden_states)

print('input_ids shape   =', tuple(input_ids.shape))
print('embeddings shape  =', tuple(embeddings.shape))
print('hidden shape      =', tuple(hidden_states.shape))
print('logits shape      =', tuple(logits.shape))
print('vocab size        =', shape_cfg.vocab_size)

del shape_model, embeddings, hidden_states, logits
gc.collect()


input_ids shape   = (2, 8)
embeddings shape  = (2, 8, 64)
hidden shape      = (2, 8, 64)
logits shape      = (2, 8, 6400)
vocab size        = 6400


24

## 3. Main Training Data Formats

MiniMind uses different JSONL structures for different training stages. Students should learn to distinguish them early.

### Pretraining

A plain text field:

- used for next-token prediction;
- no conversation structure is needed.

### Supervised Fine-Tuning (SFT)

A list of role-tagged messages:

- can contain standard dialogue;
- can contain reasoning / chain-of-thought (CoT) supervision;
- can also include tool-use interactions.

Important clarification for beginners:

MiniMind does **not** use a separate top-level 'CoT dataset format'. Instead, reasoning-style supervision is stored **inside the SFT conversation format**. The assistant keeps the final answer in `content`, and the reasoning trace in `reasoning_content`. During chat-template serialization, `reasoning_content` is converted into a `<think>...</think>` span.

### Preference Learning (for example DPO)

A pair of responses to the same prompt:

- `chosen` is the preferred answer;
- `rejected` is the less preferred answer.

### Rollout / RL-style Prompt Format

A conversation where the final assistant response may be empty:

- used when the model is expected to continue generation during rollout.


In [12]:
tool_schema = [
    {
        'type': 'function',
        'function': {
            'name': 'calculate_math',
            'description': 'Evaluate a mathematical expression.',
            'parameters': {
                'type': 'object',
                'properties': {'expression': {'type': 'string'}},
                'required': ['expression']
            }
        }
    }
]

pretrain_example = {
    'text': 'Self-attention allows a model to relate each token to the other tokens in the sequence.'
}

sft_dialog_example = {
    'conversations': [
        {'role': 'system', 'content': 'You are a concise teaching assistant.', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
        {'role': 'user', 'content': 'What is pretraining?', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
        {'role': 'assistant', 'content': 'Pretraining teaches the model general language patterns before task-specific tuning.', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
    ]
}

sft_reasoning_example = {
    'conversations': [
        {'role': 'system', 'content': 'You are a careful math tutor.', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
        {'role': 'user', 'content': 'A class has 3 rows of seats and 4 seats in each row. How many seats are there in total?', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
        {
            'role': 'assistant',
            'content': 'There are 12 seats in total.',
            'reasoning_content': 'Step 1: There are 3 rows.\nStep 2: Each row has 4 seats.\nStep 3: Multiply 3 by 4 to get 12.\nTherefore, the total number of seats is 12.',
            'tools': '',
            'tool_calls': ''
        },
    ]
}

sft_tool_example = {
    'conversations': [
        {'role': 'system', 'content': 'You may call tools when appropriate.', 'reasoning_content': '', 'tools': json.dumps(tool_schema, ensure_ascii=False), 'tool_calls': ''},
        {'role': 'user', 'content': 'Compute 12 * 13.', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
        {'role': 'assistant', 'content': '', 'reasoning_content': '', 'tools': '', 'tool_calls': json.dumps([{'name': 'calculate_math', 'arguments': {'expression': '12*13'}}], ensure_ascii=False)},
        {'role': 'tool', 'content': '{"result": 156}', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
        {'role': 'assistant', 'content': '12 * 13 = 156.', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
    ]
}

dpo_example = {
    'chosen': [
        {'role': 'user', 'content': 'What is SFT?'},
        {'role': 'assistant', 'content': 'SFT stands for supervised fine-tuning, where labeled examples guide the model toward desired behavior.'}
    ],
    'rejected': [
        {'role': 'user', 'content': 'What is SFT?'},
        {'role': 'assistant', 'content': 'I am not sure.'}
    ]
}

rlaif_style_example = {
    'conversations': [
        {'role': 'user', 'content': 'Explain attention in one sentence.'},
        {'role': 'assistant', 'content': ''}
    ]
}

print('=== Pretraining example ===')
print_json(pretrain_example)
print('\n=== Standard SFT example ===')
print_json(sft_dialog_example)
print('\n=== Reasoning / Chain-of-Thought (CoT) SFT example ===')
print_json(sft_reasoning_example)
print('\n=== Tool-use SFT example ===')
print_json(sft_tool_example)
print('\n=== DPO example ===')
print_json(dpo_example)
print('\n=== RL-style prompt example ===')
print_json(rlaif_style_example)


=== Pretraining example ===
{
  "text": "Self-attention allows a model to relate each token to the other tokens in the sequence."
}

=== Standard SFT example ===
{
  "conversations": [
    {
      "role": "system",
      "content": "You are a concise teaching assistant.",
      "reasoning_content": "",
      "tools": "",
      "tool_calls": ""
    },
    {
      "role": "user",
      "content": "What is pretraining?",
      "reasoning_content": "",
      "tools": "",
      "tool_calls": ""
    },
    {
      "role": "assistant",
      "content": "Pretraining teaches the model general language patterns before task-specific tuning.",
      "reasoning_content": "",
      "tools": "",
      "tool_calls": ""
    }
  ]
}

=== Reasoning / Chain-of-Thought (CoT) SFT example ===
{
  "conversations": [
    {
      "role": "system",
      "content": "You are a careful math tutor.",
      "reasoning_content": "",
      "tools": "",
      "tool_calls": ""
    },
    {
      "role": "user",
      "c

## 4. Tokenizer Basics and Special Control Tokens

MiniMind uses a compact tokenizer with a relatively small vocabulary and several control tokens that play an important instructional role.

Particularly important tokens include:

- `<|im_start|>` and `<|im_end|>` for chat message boundaries;
- `<think>` and `</think>` for reasoning spans;
- `<tool_call>` / `</tool_call>` and `<tool_response>` / `</tool_response>` for tool use.

The next cell inspects these tokens directly.


In [13]:
tokenizer_path = REPO_ROOT / 'minimind-3' if (REPO_ROOT / 'minimind-3').exists() else REPO_ROOT / 'model'
tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path), trust_remote_code=True)

special_tokens = [
    '<|im_start|>', '<|im_end|>',
    '<think>', '</think>',
    '<tool_call>', '</tool_call>',
    '<tool_response>', '</tool_response>'
]

print('vocab_size =', tokenizer.vocab_size)
print('bos_token =', tokenizer.bos_token, tokenizer.bos_token_id)
print('eos_token =', tokenizer.eos_token, tokenizer.eos_token_id)
print('pad_token_id =', tokenizer.pad_token_id)
print('\nSpecial token IDs:')
for tok in special_tokens:
    print(f'{tok:>18} -> {tokenizer.convert_tokens_to_ids(tok)}')


vocab_size = 6400
bos_token = <|im_start|> 1
eos_token = <|im_end|> 2
pad_token_id = 0

Special token IDs:
      <|im_start|> -> 1
        <|im_end|> -> 2
           <think> -> 25
          </think> -> 26
       <tool_call> -> 21
      </tool_call> -> 22
   <tool_response> -> 23
  </tool_response> -> 24


## 5. Concrete Tokenization Examples

Students often understand tokenizers better when they see several qualitatively different inputs side by side:

- ordinary English text;
- ordinary Chinese text;
- reasoning tags;
- tool-call tags with JSON content.

The next cell prints token IDs, token strings, and token counts for each case.

Because MiniMind uses a ByteLevel-style tokenizer, some printed token strings, especially for Chinese text, may look byte-like or visually strange. That is expected. The reliable checks are the token count and the final decoded text.


In [14]:
tokenization_examples = [
    'Hello MiniMind! Please explain transformers.',
    '你好，MiniMind！请解释一下 Transformer。',
    '<think>Reason step by step.</think>',
    '<tool_call>{"name":"calculate_math","arguments":{"expression":"12*13"}}</tool_call>'
]

for text in tokenization_examples:
    ids = tokenizer(text, add_special_tokens=False).input_ids
    toks = tokenizer.convert_ids_to_tokens(ids)
    decoded = tokenizer.decode(ids)
    print('=' * 100)
    print('TEXT:')
    print(text)
    print('TOKEN COUNT:', len(ids))
    print('TOKEN IDS (first 30):', ids[:30])
    print('TOKENS   (first 30):', toks[:30])
    print('DECODED BACK:')
    print(decoded)


TEXT:
Hello MiniMind! Please explain transformers.
TOKEN COUNT: 13
TOKEN IDS (first 30): [1602, 869, 301, 108, 80, 916, 36, 2694, 5248, 1890, 876, 496, 49]
TOKENS   (first 30): ['Hello', 'ĠM', 'in', 'i', 'M', 'ind', '!', 'ĠPlease', 'Ġexplain', 'Ġtrans', 'form', 'ers', '.']
DECODED BACK:
Hello MiniMind! Please explain transformers.
TEXT:
你好，MiniMind！请解释一下 Transformer。
TOKEN COUNT: 16
TOKEN IDS (first 30): [1968, 294, 80, 301, 108, 80, 916, 1364, 960, 2093, 2360, 527, 3289, 876, 311, 302]
TOKENS   (first 30): ['ä½łå¥½', 'ï¼Į', 'M', 'in', 'i', 'M', 'ind', 'ï¼ģ', 'è¯·', 'è§£éĩĬ', 'ä¸Ģä¸ĭ', 'ĠT', 'rans', 'form', 'er', 'ãĢĤ']
DECODED BACK:
你好，MiniMind！请解释一下 Transformer。
TEXT:
<think>Reason step by step.</think>
TOKEN COUNT: 8
TOKEN IDS (first 30): [25, 4087, 4402, 3062, 769, 3062, 49, 26]
TOKENS   (first 30): ['<think>', 'Re', 'ason', 'Ġstep', 'Ġby', 'Ġstep', '.', '</think>']
DECODED BACK:
<think>Reason step by step.</think>
TEXT:
<tool_call>{"name":"calculate_math","arguments":{"expression"

## 6. Chat Template Construction

The tokenizer is not only used to split text into tokens. In MiniMind, it also transforms structured conversation data into the exact text that the causal language model will consume.

This is one of the key teaching ideas in the notebook:

> Python dictionaries describing messages are turned into a single serialized prompt before tokenization.

The examples below show four important cases:

1. standard dialogue;
2. reasoning-style dialogue;
3. dialogue with an explicit open-thinking prompt;
4. dialogue with tool definitions.

This is also why CoT data can look 'hidden' in raw JSONL files: it first appears as `reasoning_content`, and only after template expansion does it become a visible `<think>...</think>` block.


In [15]:
formatted_plain = tokenizer.apply_chat_template(
    sft_dialog_example['conversations'],
    tokenize=False,
    add_generation_prompt=True,
)

formatted_reasoning = tokenizer.apply_chat_template(
    sft_reasoning_example['conversations'],
    tokenize=False,
    add_generation_prompt=True,
)

formatted_thinking = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': 'Explain pretraining in one sentence.'}],
    tokenize=False,
    add_generation_prompt=True,
    open_thinking=True,
)

formatted_tool = tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': 'You may call tools when appropriate.'},
        {'role': 'user', 'content': 'Compute 12 * 13.'}
    ],
    tokenize=False,
    add_generation_prompt=True,
    tools=tool_schema,
)

print('=== Standard chat template ===')
print(formatted_plain)
print('\n=== Reasoning-style chat template ===')
print(formatted_reasoning)
print('\n=== Open-thinking template ===')
print(formatted_thinking)
print('\n=== Tool-aware template (truncated) ===')
print(formatted_tool[:1000])


=== Standard chat template ===
<|im_start|>system
You are a concise teaching assistant.<|im_end|>
<|im_start|>user
What is pretraining?<|im_end|>
<|im_start|>assistant
<think>

</think>

Pretraining teaches the model general language patterns before task-specific tuning.<|im_end|>
<|im_start|>assistant
<think>

</think>



=== Reasoning-style chat template ===
<|im_start|>system
You are a careful math tutor.<|im_end|>
<|im_start|>user
A class has 3 rows of seats and 4 seats in each row. How many seats are there in total?<|im_end|>
<|im_start|>assistant
<think>
Step 1: There are 3 rows.
Step 2: Each row has 4 seats.
Step 3: Multiply 3 by 4 to get 12.
Therefore, the total number of seats is 12.
</think>

There are 12 seats in total.<|im_end|>
<|im_start|>assistant
<think>

</think>



=== Open-thinking template ===
<|im_start|>user
Explain pretraining in one sentence.<|im_end|>
<|im_start|>assistant
<think>


=== Tool-aware template (truncated) ===
<|im_start|>system
You may call tools w

## 7. Dataset Objects and Returned Tensors

A good teaching transition is to move from abstract JSON examples to actual dataset objects. This shows students what the model *really receives* during training.

The following cell builds tiny temporary JSONL files and demonstrates:

- `PretrainDataset` output;
- `SFTDataset` output for standard dialogue;
- `SFTDataset` output for reasoning / CoT-style data;
- `SFTDataset` output for tool-use data;
- `DPODataset` output.

This is the point where students can verify that reasoning data is not just a conceptual idea: it becomes ordinary token tensors, and the reasoning tokens inside the assistant span are part of the supervised targets.


In [16]:
def write_jsonl(records):
    f = tempfile.NamedTemporaryFile('w', suffix='.jsonl', delete=False, encoding='utf-8')
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
    f.close()
    return f.name

pretrain_path = write_jsonl([pretrain_example])
sft_dialog_path = write_jsonl([sft_dialog_example])
sft_reasoning_path = write_jsonl([sft_reasoning_example])
sft_tool_path = write_jsonl([sft_tool_example])
dpo_path = write_jsonl([dpo_example])

pretrain_ds = PretrainDataset(pretrain_path, tokenizer, max_length=48)
sft_dialog_ds = SFTDataset(sft_dialog_path, tokenizer, max_length=128)
sft_reasoning_ds = SFTDataset(sft_reasoning_path, tokenizer, max_length=160)
sft_tool_ds = SFTDataset(sft_tool_path, tokenizer, max_length=192)
dpo_ds = DPODataset(dpo_path, tokenizer, max_length=128)

pre_x, pre_y = pretrain_ds[0]
print('=== PretrainDataset sample ===')
print('input_ids shape =', tuple(pre_x.shape))
print('labels shape =', tuple(pre_y.shape))
valid_pre_ids = [i for i in pre_x.tolist() if i != tokenizer.pad_token_id]
print('decoded sample:')
print(tokenizer.decode(valid_pre_ids))
print('number of masked label positions =', int((pre_y == -100).sum()))

dialog_x, dialog_y = sft_dialog_ds[0]
print('\n=== Standard SFTDataset sample ===')
print('input_ids shape =', tuple(dialog_x.shape))
print('labels shape =', tuple(dialog_y.shape))
print('number of supervised positions =', int((dialog_y != -100).sum()))
print('decoded supervised target tokens:')
print(tokenizer.decode([tid for tid, lab in zip(dialog_x.tolist(), dialog_y.tolist()) if lab != -100 and tid != tokenizer.pad_token_id]))

reason_x, reason_y = sft_reasoning_ds[0]
print('\n=== Reasoning-style SFTDataset sample ===')
print('number of supervised positions =', int((reason_y != -100).sum()))
print('decoded supervised target tokens:')
print(tokenizer.decode([tid for tid, lab in zip(reason_x.tolist(), reason_y.tolist()) if lab != -100 and tid != tokenizer.pad_token_id]))

print('\n=== Tool-use SFT formatted prompt ===')
print(sft_tool_ds.create_chat_prompt(sft_tool_example['conversations'])[:1200])

dpo_sample = dpo_ds[0]
print('\n=== DPODataset sample ===')
print({k: tuple(v.shape) for k, v in dpo_sample.items()})
print('chosen supervised positions =', int(dpo_sample['mask_chosen'].sum()))
print('rejected supervised positions =', int(dpo_sample['mask_rejected'].sum()))


Generating train split: 1 examples [00:00, 403.92 examples/s]
Generating train split: 1 examples [00:00, 324.56 examples/s]
Generating train split: 1 examples [00:00, 426.51 examples/s]
Generating train split: 1 examples [00:00, 429.26 examples/s]
Generating train split: 1 examples [00:00, 543.02 examples/s]

=== PretrainDataset sample ===
input_ids shape = (48,)
labels shape = (48,)
decoded sample:
<|im_start|>Self-attention allows a model to relate each token to the other tokens in the sequence.<|im_end|>
number of masked label positions = 20

=== Standard SFTDataset sample ===
input_ids shape = (128,)
labels shape = (128,)
number of supervised positions = 27
decoded supervised target tokens:
Pretraining teaches the model general language patterns before task-specific tuning.<|im_end|>


=== Reasoning-style SFTDataset sample ===
number of supervised positions = 72
decoded supervised target tokens:
<think>
Step 1: There are 3 rows.
Step 2: Each row has 4 seats.
Step 3: Multiply 3 by 4 to get 12.
Therefore, the total number of seats is 12.
</think>

There are 12 seats in total.<|im_end|>


=== Tool-use SFT formatted prompt ===
<|im_start|>system
You may call tools when appropriate.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function si

## 8. Why SFT Does Not Supervise the Entire Sequence

A common beginner misconception is that all tokens in an SFT sample are used as learning targets. MiniMind does **not** do that.

In `SFTDataset.generate_labels(...)`, only the assistant spans are assigned valid labels. The user turns, system turns, structural markers, and padding positions are masked with `-100`, which tells PyTorch to ignore them in the loss.

Pedagogically, this is a crucial idea:

> the model **reads** the full conversation, but it is only **graded** on the assistant response.

In reasoning-style data, the reasoning tokens inside `<think>...</think>` are still part of the assistant span, so they also become supervised targets.


In [17]:
rows = []
for i, (tid, lab) in enumerate(zip(reason_x.tolist()[:100], reason_y.tolist()[:100])):
    token_text = tokenizer.decode([tid]).replace('\n', '\\n')
    rows.append((i, token_text, lab != -100, int(lab)))

print('First 40 positions from the reasoning / CoT SFT sample:')
print('(index, decoded_token, supervised, label_id)')
for row in rows[:40]:
    print(row)


First 40 positions from the reasoning / CoT SFT sample:
(index, decoded_token, supervised, label_id)
(0, '<|im_start|>', False, -100)
(1, 's', False, -100)
(2, 'ystem', False, -100)
(3, '\\n', False, -100)
(4, 'You', False, -100)
(5, ' are', False, -100)
(6, ' a', False, -100)
(7, ' c', False, -100)
(8, 'are', False, -100)
(9, 'ful', False, -100)
(10, ' math', False, -100)
(11, ' t', False, -100)
(12, 'ut', False, -100)
(13, 'or', False, -100)
(14, '.', False, -100)
(15, '<|im_end|>', False, -100)
(16, '\\n', False, -100)
(17, '<|im_start|>', False, -100)
(18, 'us', False, -100)
(19, 'er', False, -100)
(20, '\\n', False, -100)
(21, 'A', False, -100)
(22, ' class', False, -100)
(23, ' has', False, -100)
(24, ' 3', False, -100)
(25, ' ro', False, -100)
(26, 'ws', False, -100)
(27, ' of', False, -100)
(28, ' se', False, -100)
(29, 'ats', False, -100)
(30, ' and', False, -100)
(31, ' 4', False, -100)
(32, ' se', False, -100)
(33, 'ats', False, -100)
(34, ' in', False, -100)
(35, ' each', F

## 9. Loss Functions

This section connects the mathematics to the implementation.

For beginners, Sections 9.1 and 9.2 are the essential material. Sections 9.3 and 9.4 are useful extensions that show how the same codebase grows beyond plain next-token prediction.

### 9.1 Pretraining Objective

For a token sequence $x_1, x_2, \dots, x_T$, the autoregressive objective is

$$
\mathcal{L}_{\text{pretrain}}(\theta)
= - \sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t}).
$$

### 9.2 Masked SFT Objective

Let $m_t \in \{0,1\}$ indicate whether token position $t$ belongs to the assistant target span. Then the masked SFT objective is

$$
\mathcal{L}_{\text{SFT}}(\theta)
= - \frac{1}{\sum_t m_t} \sum_{t=1}^{T} m_t \, \log p_\theta(y_t \mid x_{\le t}).
$$

In MiniMind, this masking is implemented with `ignore_index=-100` in cross-entropy.

### 9.3 Total Loss with MoE Auxiliary Term

When the MoE variant is used, the training loop adds the routing auxiliary loss:

$$
\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{CE}} + \mathcal{L}_{\text{aux}}.
$$

### 9.4 Preference Learning (DPO)

For preferred answer $y^+$ and rejected answer $y^-$, a standard DPO-style objective is

$$
\mathcal{L}_{\text{DPO}}(\theta)
= - \log \sigma \Big(
\beta \big[
(\log \pi_\theta(y^+|x) - \log \pi_\theta(y^-|x))
-
(\log \pi_{\text{ref}}(y^+|x) - \log \pi_{\text{ref}}(y^-|x))
\big]
\Big).
$$

For students coming from standard deep learning, the key message is simple:

- the **core** loss is still cross-entropy;
- what changes is how targets are masked and how different training stages define those targets.


## 10. Verifying the Masked Cross-Entropy in Code

The following cell constructs a very small MiniMind model and verifies that:

- the model's own reported loss;
- a manually recomputed cross-entropy loss with `ignore_index=-100`

match each other for a toy SFT batch.

A second miniature MoE model is also created to show how the auxiliary loss appears separately.


In [18]:
demo_cfg = MiniMindConfig(
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    vocab_size=6400,
    max_position_embeddings=256,
    use_moe=False,
)

demo_model = MiniMindForCausalLM(demo_cfg)
batch_input = reason_x[:96].unsqueeze(0)
batch_labels = reason_y[:96].unsqueeze(0)
res = demo_model(batch_input, labels=batch_labels)

logits = res.logits[..., :-1, :].contiguous()
targets = batch_labels[..., 1:].contiguous()
manual_loss = F.cross_entropy(
    logits.view(-1, logits.size(-1)),
    targets.view(-1),
    ignore_index=-100,
)

print('model loss        =', float(res.loss))
print('manual CE loss    =', float(manual_loss))
print('absolute diff     =', abs(float(res.loss) - float(manual_loss)))
print('auxiliary loss    =', float(res.aux_loss))

moe_cfg = MiniMindConfig(
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    vocab_size=6400,
    max_position_embeddings=256,
    use_moe=True,
    num_experts=4,
    num_experts_per_tok=1,
)

moe_model = MiniMindForCausalLM(moe_cfg)
moe_res = moe_model(batch_input[:, :64], labels=batch_labels[:, :64])

print('\n=== MoE demonstration ===')
print('cross-entropy loss =', float(moe_res.loss))
print('auxiliary loss     =', float(moe_res.aux_loss))
print('total loss         =', float(moe_res.loss + moe_res.aux_loss))

del demo_model, moe_model, res, moe_res
gc.collect()


model loss        = 8.844755172729492
manual CE loss    = 8.844755172729492
absolute diff     = 0.0
auxiliary loss    = 0.0

=== MoE demonstration ===
cross-entropy loss = 8.840554237365723
auxiliary loss     = 0.001059270929545164
total loss         = 8.84161376953125


33

## 11. Model Skeleton

At the source-code level, the main architecture is organized as follows:

1. **`MiniMindConfig`** defines model size and architectural options;
2. **`Attention`** implements grouped-query causal self-attention with RoPE;
3. **`FeedForward`** or **`MOEFeedForward`** provides the per-token MLP stage;
4. **`MiniMindBlock`** combines attention, MLP, normalization, and residual connections;
5. **`MiniMindModel`** stacks blocks on top of token embeddings;
6. **`MiniMindForCausalLM`** adds the language-model head and computes next-token loss.

The code below instantiates a deliberately small dense demonstration model so that students can inspect the structure without loading the released checkpoint.


In [19]:
structure_cfg = MiniMindConfig(
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    vocab_size=6400,
    max_position_embeddings=256,
    use_moe=False,
)

structure_model = MiniMindForCausalLM(structure_cfg)
structure_input = torch.randint(0, structure_cfg.vocab_size, (2, 16))
structure_out = structure_model(structure_input)

print('logits shape =', tuple(structure_out.logits.shape))
print('parameter count = %.2fM' % (count_params(structure_model) / 1e6))
print('\nFirst transformer block:')
print(structure_model.model.layers[0])

export_config_path = REPO_ROOT / 'minimind-3' / 'config.json'
if export_config_path.exists():
    export_cfg = json.loads(export_config_path.read_text(encoding='utf-8'))
    print('\nSelected fields from the exported Transformers config:')
    keep = ['model_type', 'hidden_size', 'num_hidden_layers', 'num_attention_heads', 'num_key_value_heads', 'vocab_size']
    print_json({k: export_cfg[k] for k in keep})
    print('\nTeaching note: the exported checkpoint is aligned with the Qwen-style ecosystem, while the repository also includes a native MiniMind implementation for study.')

del structure_model, structure_out
gc.collect()


logits shape = (2, 16, 6400)
parameter count = 1.26M

First transformer block:
MiniMindBlock(
  (self_attn): Attention(
    (q_proj): Linear(in_features=128, out_features=128, bias=False)
    (k_proj): Linear(in_features=128, out_features=64, bias=False)
    (v_proj): Linear(in_features=128, out_features=64, bias=False)
    (o_proj): Linear(in_features=128, out_features=128, bias=False)
    (q_norm): RMSNorm()
    (k_norm): RMSNorm()
    (attn_dropout): Dropout(p=0.0, inplace=False)
    (resid_dropout): Dropout(p=0.0, inplace=False)
  )
  (input_layernorm): RMSNorm()
  (post_attention_layernorm): RMSNorm()
  (mlp): FeedForward(
    (gate_proj): Linear(in_features=128, out_features=448, bias=False)
    (down_proj): Linear(in_features=448, out_features=128, bias=False)
    (up_proj): Linear(in_features=128, out_features=448, bias=False)
    (act_fn): SiLUActivation()
  )
)

Selected fields from the exported Transformers config:
{
  "model_type": "qwen3",
  "hidden_size": 768,
  "num_hidd

33

## 12. Source-Code Anchors

The following cell prints a small set of carefully chosen source snippets. In class, these are often enough to connect the mathematical and conceptual discussion back to the real implementation.

Recommended points of emphasis:

- `PretrainDataset.__getitem__`;
- `SFTDataset.create_chat_prompt` and `SFTDataset.generate_labels`;
- `MiniMindConfig` and `Attention`;
- `MiniMindForCausalLM.forward`;
- the main loops in `train_pretrain.py` and `train_full_sft.py`.


In [20]:
print('=== Source snippet: PretrainDataset ===')
show_snippet(REPO_ROOT / 'dataset' / 'lm_dataset.py', 37, 55)

print('\n=== Source snippet: SFTDataset ===')
show_snippet(REPO_ROOT / 'dataset' / 'lm_dataset.py', 58, 119)

print('\n=== Source snippet: Config and attention ===')
show_snippet(REPO_ROOT / 'model' / 'model_minimind.py', 10, 45)
show_snippet(REPO_ROOT / 'model' / 'model_minimind.py', 91, 134)

print('\n=== Source snippet: Causal LM forward ===')
show_snippet(REPO_ROOT / 'model' / 'model_minimind.py', 234, 253)

print('\n=== Source snippet: pretraining main flow ===')
show_snippet(REPO_ROOT / 'trainer' / 'train_pretrain.py', 82, 166)

print('\n=== Source snippet: SFT main flow ===')
show_snippet(REPO_ROOT / 'trainer' / 'train_full_sft.py', 83, 167)


=== Source snippet: PretrainDataset ===
  37: class PretrainDataset(Dataset):
  38:     def __init__(self, data_path, tokenizer, max_length=512):
  39:         super().__init__()
  40:         self.tokenizer = tokenizer
  41:         self.max_length = max_length
  42:         self.samples = load_dataset('json', data_files=data_path, split='train')
  43: 
  44:     def __len__(self):
  45:         return len(self.samples)
  46: 
  47:     def __getitem__(self, index):
  48:         sample = self.samples[index]
  49:         tokens = self.tokenizer(str(sample['text']), add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
  50:         tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
  51:         input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
  52:         input_ids = torch.tensor(input_ids, dtype=torch.long)
  53:         labels = input_ids.clone()
  54:         labels[input_ids == self.

## 13. Optional Real Inference Demonstration

The following cell is disabled by default. In a lecture setting, it is often better to keep the notebook lightweight and only enable real inference when students are ready to connect the abstract pipeline to an actual model response.

Set `RUN_REAL_INFERENCE = True` only if the local `minimind-3` model directory is available and you want to run a live demo.


In [21]:
RUN_REAL_INFERENCE = False

if RUN_REAL_INFERENCE:
    from transformers import AutoModelForCausalLM

    model_path = REPO_ROOT / 'minimind-3'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = AutoModelForCausalLM.from_pretrained(str(model_path), trust_remote_code=True)
    if device == 'cuda':
        model = model.half().eval().to(device)
    else:
        model = model.eval().to(device)

    prompt = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': 'Explain pretraining in one sentence.'}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    outputs = model.generate(**inputs, max_new_tokens=64, do_sample=True, top_p=0.95, temperature=0.85)
    answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(answer)
else:
    print('Set RUN_REAL_INFERENCE = True to run a live generation demo.')


Set RUN_REAL_INFERENCE = True to run a live generation demo.


## 14. Teaching Takeaways and Discussion Questions

For classroom use, the most important conceptual messages are:

- MiniMind is best understood as a **pipeline**, not just as a single model file.
- Data structure matters: pretraining text, standard SFT dialogue, reasoning traces, tool-use traces, and preference pairs serve different purposes.
- MiniMind does not use a separate top-level CoT file format; reasoning traces are stored inside the SFT assistant message through `reasoning_content`.
- The tokenizer does more than token splitting; it also participates in prompt serialization through the chat template.
- In SFT, the model reads the full conversation but is supervised mainly on assistant output.
- Reasoning-style data becomes a `<think>...</think>` span after template construction, and those reasoning tokens can still contribute to the loss.
- The source code is compact enough that students can trace the path from dataset to logits and loss.

### Suggested Reading Order After This Notebook

1. `dataset/lm_dataset.py`
2. `model/model_minimind.py`
3. `trainer/train_pretrain.py`
4. `trainer/train_full_sft.py`
5. `eval_llm.py`

This sequence usually gives students the clearest conceptual progression from data to model to training to generation.
